# RL graph scheduling experiment

Run this notebook from top to bottom. It uses the pinned project evaluator and trains a separate small REINFORCE policy for each graph. All three methods below use the same evaluator and baseline schedule. The scores here are **local modeled latencies**, not official contest points.


## 1. Setup

Run from either the repository root or `rl_scheduler/`. Change `BENCHMARK_IDS` or `EPISODES` here and rerun all cells. The fixed seed makes repeated runs comparable on the same machine and package versions.


In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import random
import sys

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import torch
from torch import nn
from torch.distributions import Categorical

ROOT = Path.cwd().resolve()
if not (ROOT / "benchmarks").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "benchmarks").is_dir():
    raise RuntimeError("Run this notebook from the repository root or rl_scheduler/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from evaluator import Evaluator

SEED = 42
BENCHMARK_IDS = (1, 5, 9, 13, 17)
EPISODES = 300
LEARNING_RATE = 1e-3
ENTROPY_COEF = 1e-3

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)

set_seed(SEED)
print(f"Repository: {ROOT}")
print(f"Benchmarks: {BENCHMARK_IDS}; episodes per graph: {EPISODES}; seed: {SEED}")


Repository: /Users/hoangnamtran/Desktop/STUDY/Junior/CIS4930/project/usf-cis4930
Benchmarks: (1, 5, 9); episodes per graph: 300; seed: 42


## 2. Scheduling environment

The agent examines boundaries in a valid topological baseline schedule. Action `0` keeps the boundary; action `1` fuses the two adjacent groups if the resulting full schedule is valid. Invalid fusions are masked. This preserves the notebook's original action space and trial tile (`[128, 128, 64]` for a group with MatMul, otherwise `[128, 128, 1]`). Fusion clears retention for the merged group. The observation includes current group indices, boundary position, and four hardware scalars; it does **not** yet encode graph edges, tensor sizes, or operation costs.


In [2]:
class SchedulingEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, problem):
        super().__init__()
        self.problem = deepcopy(problem)
        self.evaluator = Evaluator(self.problem)
        baseline = self.evaluator.baseline()
        self.baseline_schedule = deepcopy(baseline["schedule"])
        self.baseline_schedule.pop("subgraph_latencies", None)
        self.baseline_latency = float(baseline["evaluation"]["total_latency"])
        self.num_ops = len(self.problem["op_types"])
        self.observation_space = spaces.Box(
            low=0.0, high=np.inf, shape=(self.num_ops + 5,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(2)
        self.schedule = None
        self.boundary_index = 0
        self.terminated = False

    def _observation(self):
        obs = np.zeros(self.num_ops + 5, dtype=np.float32)
        scale = max(1, self.num_ops - 1)
        for group_index, group in enumerate(self.schedule["subgraphs"]):
            for op in group:
                obs[op] = group_index / scale
        obs[self.num_ops] = self.boundary_index / scale
        obs[self.num_ops + 1:] = [
            self.problem["fast_memory_capacity"] / 1_000_000,
            self.problem["slow_memory_bandwidth"] / 100,
            self.problem["native_granularity"][0] / 128,
            self.problem["native_granularity"][1] / 128,
        ]
        return obs

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.schedule = deepcopy(self.baseline_schedule)
        self.boundary_index = 0
        self.terminated = len(self.schedule["subgraphs"]) <= 1
        return self._observation(), {"baseline_latency": self.baseline_latency}

    def fusion_candidate(self):
        if self.terminated:
            raise RuntimeError("Episode ended; call reset().")
        candidate = deepcopy(self.schedule)
        i = self.boundary_index
        merged = candidate["subgraphs"][i] + candidate["subgraphs"][i + 1]
        has_matmul = any(self.problem["op_types"][op] == "MatMul" for op in merged)
        candidate["subgraphs"][i:i + 2] = [merged]
        candidate["granularities"][i:i + 2] = [[128, 128, 64 if has_matmul else 1]]
        candidate["tensors_to_retain"][i:i + 2] = [[]]
        if "traversal_orders" in candidate:
            candidate["traversal_orders"][i:i + 2] = [None]
        return candidate

    def action_mask(self):
        if self.terminated:
            return np.array([True, False], dtype=bool)
        result = self.evaluator.evaluate(self.fusion_candidate())
        return np.array([True, bool(result["valid"])], dtype=bool)

    def step(self, action):
        if self.terminated:
            raise RuntimeError("Episode ended; call reset().")
        if not self.action_space.contains(action):
            raise ValueError("Action must be KEEP (0) or FUSE (1).")
        if action == 0:
            self.boundary_index += 1
        else:
            candidate = self.fusion_candidate()
            result = self.evaluator.evaluate(candidate)
            if not result["valid"]:
                self.terminated = True
                return self._observation(), -1.0, True, False, result
            self.schedule = candidate
        self.terminated = self.boundary_index >= len(self.schedule["subgraphs"]) - 1
        if not self.terminated:
            return self._observation(), 0.0, False, False, {}
        feedback = self.evaluator.reward(self.schedule)
        info = dict(feedback["evaluation"])
        info["baseline_latency"] = self.baseline_latency
        return self._observation(), float(feedback["reward"]), True, False, info

    def close(self):
        self.evaluator.close()


## 3. Baseline and greedy comparator

Greedy tests the candidate fusion at each boundary and accepts it only when the **current full schedule's** modeled latency falls. It is a simple reference for the current action space, not an optimal solver.


In [3]:
def run_greedy(problem):
    env = SchedulingEnv(problem)
    try:
        env.reset()
        actions = []
        current_latency = env.baseline_latency
        while not env.terminated:
            candidate = env.fusion_candidate()
            result = env.evaluator.evaluate(candidate)
            fuse = bool(result["valid"] and result["total_latency"] < current_latency - 1e-9)
            action = int(fuse)
            _, reward, _, _, info = env.step(action)
            actions.append(action)
            if fuse:
                current_latency = float(result["total_latency"])
        final = env.evaluator.evaluate(env.schedule)
        assert final["valid"]
        return {
            "latency": float(final["total_latency"]),
            "reward": float(reward) if actions else 0.5,
            "actions": actions,
            "schedule": deepcopy(env.schedule),
        }
    finally:
        env.close()


## 4. REINFORCE policy

Each benchmark gets its own policy because the current observation length depends on the graph's operation count. Training uses masked actions, a moving reward baseline, and a small entropy bonus for exploration. The saved result reports both the deterministic final policy and the best sampled episode; the latter is a search result, not the policy's average performance.


In [4]:
def train_reinforce(problem, *, seed, episodes):
    set_seed(seed)
    env = SchedulingEnv(problem)
    policy = nn.Sequential(
        nn.Linear(env.observation_space.shape[0], 64),
        nn.Tanh(),
        nn.Linear(64, env.action_space.n),
    )
    optimizer = torch.optim.Adam(policy.parameters(), lr=LEARNING_RATE)
    moving_reward = 0.5
    best = {"reward": 0.5, "latency": env.baseline_latency,
            "schedule": deepcopy(env.baseline_schedule)}
    history = []
    try:
        for episode in range(episodes):
            obs, _ = env.reset(seed=seed + episode)
            log_probs = []
            entropies = []
            reward = 0.5
            while not env.terminated:
                logits = policy(torch.as_tensor(obs, dtype=torch.float32))
                mask = torch.as_tensor(env.action_mask(), dtype=torch.bool)
                distribution = Categorical(logits=logits.masked_fill(~mask, -torch.inf))
                action = distribution.sample()
                log_probs.append(distribution.log_prob(action))
                entropies.append(distribution.entropy())
                obs, reward, _, _, info = env.step(int(action.item()))
            if log_probs:
                advantage = reward - moving_reward
                loss = -torch.stack(log_probs).sum() * advantage
                loss -= ENTROPY_COEF * torch.stack(entropies).sum()
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            moving_reward = 0.95 * moving_reward + 0.05 * reward
            history.append(float(reward))
            if reward > best["reward"]:
                best = {"reward": float(reward),
                        "latency": float(info["total_latency"]),
                        "schedule": deepcopy(env.schedule)}
        return policy, history, best
    finally:
        env.close()


def run_policy(problem, policy):
    env = SchedulingEnv(problem)
    try:
        obs, _ = env.reset()
        actions = []
        reward = 0.5
        with torch.no_grad():
            while not env.terminated:
                logits = policy(torch.as_tensor(obs, dtype=torch.float32))
                mask = torch.as_tensor(env.action_mask(), dtype=torch.bool)
                action = int(logits.masked_fill(~mask, -torch.inf).argmax().item())
                actions.append(action)
                obs, reward, _, _, _ = env.step(action)
        final = env.evaluator.evaluate(env.schedule)
        assert final["valid"]
        return {"latency": float(final["total_latency"]),
                "reward": float(reward), "actions": actions,
                "schedule": deepcopy(env.schedule)}
    finally:
        env.close()


## 5. Run the same comparison on each graph

`latency/baseline` is lower when the schedule is faster. `reward` uses the project evaluator's bounded formula, so the baseline is always `0.5`. Results and evaluator model identifiers are saved under `build/` for a repeatable record. The baseline, greedy, and trained policy each get a fresh environment.


In [5]:
results = []
for benchmark_id in BENCHMARK_IDS:
    problem_path = ROOT / "benchmarks" / f"mlsys-2026-{benchmark_id}.json"
    problem = json.loads(problem_path.read_text())
    baseline_env = SchedulingEnv(problem)
    try:
        baseline_latency = baseline_env.baseline_latency
        model_id = baseline_env.evaluator.configuration["model_id"]
    finally:
        baseline_env.close()
    greedy = run_greedy(problem)
    policy, history, best = train_reinforce(
        problem, seed=SEED + benchmark_id, episodes=EPISODES
    )
    trained = run_policy(problem, policy)
    row = {
        "benchmark": benchmark_id, "num_ops": len(problem["op_types"]),
        "seed": SEED + benchmark_id, "episodes": EPISODES,
        "model_id": model_id, "baseline_latency": baseline_latency,
        "greedy_latency": greedy["latency"],
        "greedy_reward": greedy["reward"],
        "greedy_actions": greedy["actions"],
        "trained_latency": trained["latency"],
        "trained_reward": trained["reward"],
        "trained_actions": trained["actions"],
        "best_sampled_latency": best["latency"],
        "best_sampled_reward": best["reward"],
        "last_50_training_mean_reward": float(np.mean(history[-50:])),
    }
    results.append(row)
    print(
        f"#{benchmark_id:02d} ({row['num_ops']} ops) | "
        f"baseline {baseline_latency:.2f} | "
        f"greedy {greedy['latency'] / baseline_latency:.3f}x, "
        f"policy {trained['latency'] / baseline_latency:.3f}x, "
        f"best sampled {best['latency'] / baseline_latency:.3f}x"
    )

output_path = ROOT / "build" / "rl-notebook-results.json"
output_path.parent.mkdir(exist_ok=True)
output_path.write_text(json.dumps(results, indent=2) + "\n")
print(f"Saved results to {output_path}")


#01 (5 ops) | baseline 419430.40 | greedy 0.938x, policy 0.938x, best sampled 0.938x
#05 (19 ops) | baseline 1336516.27 | greedy 1.000x, policy 1.000x, best sampled 1.000x
#09 (32 ops) | baseline 48318382.08 | greedy 0.931x, policy 0.931x, best sampled 0.931x
Saved results to /Users/hoangnamtran/Desktop/STUDY/Junior/CIS4930/project/usf-cis4930/build/rl-notebook-results.json


## 6. Read the result

Look at `trained_latency / baseline_latency` for policy performance on each graph. The greedy and best-sampled columns show what the restricted merge action can find. This experiment trains on each tested graph, so it does **not** measure transfer to unseen graphs. A future graph-based policy can address variable graph sizes and richer structure after these results are stable.


In [6]:
for row in results:
    ratio = row["trained_latency"] / row["baseline_latency"]
    improvement = (1 - ratio) * 100
    print(
        f"Graph {row['benchmark']:02d}: trained reward {row['trained_reward']:.4f}, "
        f"latency change {improvement:+.2f}%, "
        f"greedy reward {row['greedy_reward']:.4f}, "
        f"best sampled reward {row['best_sampled_reward']:.4f}"
    )


Graph 01: trained reward 0.5161, latency change +6.25%, greedy reward 0.5161, best sampled reward 0.5161
Graph 05: trained reward 0.5000, latency change +0.00%, greedy reward 0.5000, best sampled reward 0.5000
Graph 09: trained reward 0.5180, latency change +6.94%, greedy reward 0.5180, best sampled reward 0.5180
